# Agentic IFRS S1/S2 Report Generation Pipeline

This notebook generates IFRS S1/S2 report sections from synthetic company payloads and the deterministic IFRS requirements KB.

Core design:

- **Content authority:** IFRS requirements + company payload only.
- **Authoring style:** extracted style artifacts under `style_system/authoring`.
- **Missing requirements:** stored in JSON audit files, **not mentioned in the report**.
- **Safety spine:** deterministic claims integrity, factlock, reference firewall, and approval gate.
- **LLM layer:** writer, claims register builder, judges, reviser, and optional fuzzy evidence mapper.
- **PDF layout:** excluded from drafting. `layout_style_guide.json` is used only by the separate PDF assembly stage.

In [1]:
# ============================================================
# CELL 1 — SETUP PATHS AND CONFIG
# Notebook expected location: /notebooks
# Style system expected at : /notebooks/gen_data/style/style_system
# ============================================================

import os
import re
import json
import time
import uuid
import shutil
import random
import urllib.request
import urllib.error
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict, Counter

import pandas as pd

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    raise ImportError("Install python-dotenv first: pip install python-dotenv")

load_dotenv(find_dotenv(), override=False)

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOK_DIR = (CURRENT_DIR / "notebooks").resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR

GEN_DATA_DIR = NOTEBOOK_DIR / "gen_data"

# Input folders. Override with env vars if your structure differs.
PAYLOAD_DIR = Path(os.getenv("PAYLOAD_DIR", GEN_DATA_DIR / "payloads")).resolve()
REQUIREMENTS_DIR = Path(os.getenv("IFRS_REQUIREMENTS_DIR", GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final")).resolve()
STYLE_SYSTEM_DIR = Path(os.getenv("STYLE_SYSTEM_DIR", GEN_DATA_DIR / "style" / "style_system")).resolve()

# Output folder.
OUTPUT_DIR = Path(os.getenv("GENERATION_OUTPUT_DIR", GEN_DATA_DIR / "generated_reports" / "agentic_ifrs_report")).resolve()

# Pipeline controls.
PIPELINE_MODE = os.getenv("PIPELINE_MODE", "synthetic_demo")
FORBID_INVENTION = True  # hard invariant, not a configurable switch
ALLOW_PARTIAL_COVERAGE = os.getenv("ALLOW_PARTIAL_COVERAGE", "true").lower() == "true"
USE_FUZZY_EVIDENCE_MAPPER = os.getenv("USE_FUZZY_EVIDENCE_MAPPER", "false").lower() == "true"
MAX_REVISION_LOOPS = int(os.getenv("MAX_REVISION_LOOPS", "2"))

# Section order used by the final report.
SECTIONS = [
    "General Requirements",
    "Governance",
    "Strategy",
    "Risk Management",
    "Metrics and Targets",
]

SECTION_SLUGS = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}

# Output subfolders.
DIRS = {
    "evidence_maps": OUTPUT_DIR / "01_evidence_maps",
    "coverage": OUTPUT_DIR / "02_coverage",
    "missing_requirements": OUTPUT_DIR / "03_missing_requirements",
    "plans": OUTPUT_DIR / "04_disclosure_plans",
    "drafts": OUTPUT_DIR / "05_draft_sections",
    "claims": OUTPUT_DIR / "06_claims_registers",
    "gates": OUTPUT_DIR / "07_deterministic_gates",
    "judges": OUTPUT_DIR / "08_judge_results",
    "revisions": OUTPUT_DIR / "09_revised_sections",
    "approved": OUTPUT_DIR / "10_approved_sections",
    "connectivity": OUTPUT_DIR / "11_connectivity",
    "handoff": OUTPUT_DIR / "12_pdf_handoff",
    "audit_logs": OUTPUT_DIR / "audit_logs",
}

for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("Current working directory:", CURRENT_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Payload directory:", PAYLOAD_DIR)
print("Requirements directory:", REQUIREMENTS_DIR)
print("Style system directory:", STYLE_SYSTEM_DIR)
print("Output directory:", OUTPUT_DIR)
print("Pipeline mode:", PIPELINE_MODE)
print("Forbid invention:", FORBID_INVENTION)
print("Use fuzzy mapper:", USE_FUZZY_EVIDENCE_MAPPER)

Current working directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Notebook directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Payload directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads
Requirements directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final
Style system directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system
Output directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report
Pipeline mode: synthetic_demo
Forbid invention: True
Use fuzzy mapper: False


## Azure/OpenAI helper

This patched client supports **two Azure configuration styles**.

Recommended `.env` style:

```env
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_ENDPOINT=https://<your-resource-name>.openai.azure.com
AZURE_OPENAI_STRONG_DEPLOYMENT=<your-strong-deployment-name>
AZURE_OPENAI_FAST_DEPLOYMENT=<your-fast-deployment-name>
AZURE_OPENAI_API_VERSION=2024-10-21
```

Alternative full URL style:

```env
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_STRONG_DEPLOYMENT_URL=https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=2024-10-21
AZURE_OPENAI_FAST_DEPLOYMENT_URL=https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=2024-10-21
```

Important: for Azure, the deployment value must be the **Azure deployment name**, not just the model family name unless your deployment is actually named that way.


In [2]:
# ============================================================
# CELL 2 — LLM CLIENT
# Full-URL Azure logic, matching the working role-based REST style.
# Uses:
# - AZURE_OPENAI_GPT52_DEPLOYMENT_URL for strong agents
# - AZURE_OPENAI_FAST_DEPLOYMENT_URL for fast/light agents
# ============================================================

import http.client

AZURE_OPENAI_API_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)

AZURE_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")

# Full Azure chat-completions deployment URLs.
# These must be full deployment URLs, not deployment names.
AZURE_OPENAI_GPT52_DEPLOYMENT_URL = os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")
AZURE_OPENAI_FAST_DEPLOYMENT_URL = os.getenv("AZURE_OPENAI_FAST_DEPLOYMENT_URL")


def _clean_url(value: Optional[str]) -> Optional[str]:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_GPT52_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_GPT52_DEPLOYMENT_URL)
AZURE_OPENAI_FAST_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_FAST_DEPLOYMENT_URL)

# Fast deployment falls back to GPT-5.2 if no fast endpoint is configured.
if not AZURE_OPENAI_FAST_DEPLOYMENT_URL:
    AZURE_OPENAI_FAST_DEPLOYMENT_URL = AZURE_OPENAI_GPT52_DEPLOYMENT_URL


MODEL_CONFIG = {
    "fuzzy_evidence_mapper": "fast",
    "section_writer": "strong",
    "claims_register_builder": "strong",
    "ifrs_coverage_judge": "strong",
    "evidence_judge": "strong",
    "style_judge": "fast",
    "minimal_reviser": "strong",
    "whole_report_connectivity_judge": "strong",
}


def _with_api_version(url: Optional[str]) -> Optional[str]:
    """Ensure full Azure URL has api-version query parameter."""
    if not url:
        return None
    if "api-version=" in url:
        return url
    joiner = "&" if "?" in url else "?"
    return f"{url}{joiner}api-version={AZURE_API_VERSION}"


def _mask_url_for_display(url: Optional[str]) -> str:
    """Mask resource/deployment details while keeping useful diagnostics."""
    if not url:
        return "NOT CONFIGURED"

    try:
        import urllib.parse
        parsed = urllib.parse.urlparse(url)
        host = parsed.netloc
        if host:
            host_parts = host.split(".")
            if host_parts and len(host_parts[0]) > 6:
                host_parts[0] = host_parts[0][:3] + "***" + host_parts[0][-2:]
            host = ".".join(host_parts)

        path = parsed.path
        path = re.sub(
            r"(/openai/deployments/)([^/]+)(/chat/completions)",
            lambda m: m.group(1) + m.group(2)[:2] + "***" + m.group(3),
            path,
        )
        query = "api-version=..." if parsed.query else ""
        return urllib.parse.urlunparse((parsed.scheme, host, path, "", query, ""))
    except Exception:
        return "<configured URL, masking failed>"


def _validate_full_deployment_url(name: str, url: Optional[str]) -> None:
    if not url:
        raise ValueError(
            f"Missing {name}.\n\n"
            "Required .env format:\n"
            "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
            "AZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full GPT-5.2 deployment URL>\n"
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL=<full fast deployment URL>\n\n"
            "Full URL shape:\n"
            "https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=..."
        )

    if not url.startswith("https://"):
        raise ValueError(f"{name} must be a full HTTPS Azure deployment URL: {url!r}")

    if "/openai/deployments/" not in url or "/chat/completions" not in url:
        raise ValueError(
            f"{name} does not look like a full Azure chat-completions deployment URL.\n"
            f"Configured URL shape: {_mask_url_for_display(url)}\n\n"
            "Expected shape:\n"
            "https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=...\n\n"
            "Do not put only the model name or deployment name here."
        )


AZURE_OPENAI_GPT52_DEPLOYMENT_URL = _with_api_version(AZURE_OPENAI_GPT52_DEPLOYMENT_URL)
AZURE_OPENAI_FAST_DEPLOYMENT_URL = _with_api_version(AZURE_OPENAI_FAST_DEPLOYMENT_URL)


def validate_llm_config() -> None:
    if not AZURE_OPENAI_API_KEY:
        raise ValueError(
            "Missing AZURE_OPENAI_API_KEY / OPENAI_API_KEY in .env.\n"
            "Use the shared Azure resource key if both deployments belong to the same resource."
        )

    _validate_full_deployment_url(
        "AZURE_OPENAI_GPT52_DEPLOYMENT_URL",
        AZURE_OPENAI_GPT52_DEPLOYMENT_URL,
    )
    _validate_full_deployment_url(
        "AZURE_OPENAI_FAST_DEPLOYMENT_URL",
        AZURE_OPENAI_FAST_DEPLOYMENT_URL,
    )


validate_llm_config()

print("Azure OpenAI full-URL configuration loaded")
print("Strong endpoint (GPT-5.2):", _mask_url_for_display(AZURE_OPENAI_GPT52_DEPLOYMENT_URL))
print("Fast endpoint:", _mask_url_for_display(AZURE_OPENAI_FAST_DEPLOYMENT_URL))


def get_model_url(model_tier: str = "strong") -> str:
    model_tier = (model_tier or "strong").lower().strip()
    if model_tier == "fast":
        return AZURE_OPENAI_FAST_DEPLOYMENT_URL
    return AZURE_OPENAI_GPT52_DEPLOYMENT_URL


def _extract_message_content(data: Dict[str, Any]) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )
    return content


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: List[Dict[str, str]],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: Optional[float] = None,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 6,
) -> Dict[str, Any]:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Uses full Azure chat-completions deployment URLs.
    - Retries transient 429/500/502/503/504 and connection errors.
    - First tries max_completion_tokens for GPT-5.x gateways.
    - If unsupported, falls back to max_tokens.
    - Does not expose API keys in errors.
    """

    token_fields = ["max_completion_tokens", "max_tokens"]
    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                if exc.code == 404:
                    raise RuntimeError(
                        f"{request_label} HTTP 404 Resource not found.\n"
                        f"Endpoint: {_mask_url_for_display(url)}\n\n"
                        "This usually means the full deployment URL is wrong, the deployment name is wrong, "
                        "or the deployment is not in that Azure OpenAI resource.\n\n"
                        "Check these .env variables:\n"
                        "AZURE_OPENAI_GPT52_DEPLOYMENT_URL\n"
                        "AZURE_OPENAI_FAST_DEPLOYMENT_URL"
                    ) from exc

                rate_limited = exc.code == 429
                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if rate_limited and attempt < max_attempts:
                    retry_after = None
                    try:
                        ra = exc.headers.get("Retry-After") if exc.headers else None
                        if ra is not None:
                            retry_after = float(str(ra).strip())
                    except (TypeError, ValueError):
                        retry_after = None

                    wait = retry_after if retry_after is not None else (2 ** attempt) * 2 + random.random()
                    wait = min(wait, 90)
                    print(
                        f"{request_label}: rate limited (429); retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support max_completion_tokens; "
                        "retrying with max_tokens."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

            except (ConnectionError, TimeoutError, OSError, http.client.RemoteDisconnected) as exc:
                last_error = RuntimeError(
                    f"{request_label} connection reset/timeout.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Error: {type(exc).__name__}: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection reset/timeout ({type(exc).__name__}); "
                        f"retrying attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(f"{request_label} request failed for an unknown reason.")


def azure_chat(
    messages: List[Dict[str, str]],
    model_tier: str = "strong",
    temperature: float = 0.0,
    max_tokens: int = 4000,
    response_format: Optional[Dict[str, str]] = None,
    retries: int = 6,
    retry_sleep: float = 2.0,  # kept for backward compatibility with old call sites
) -> str:
    """Azure Chat Completions helper used by all LLM agents in the notebook."""
    url = get_model_url(model_tier)
    request_label = f"Azure {model_tier} agent"

    data = _azure_chat_completion(
        url=url,
        api_key=AZURE_OPENAI_API_KEY,
        messages=messages,
        max_output_tokens=max_tokens,
        json_mode=bool(response_format),
        temperature=temperature,
        request_label=request_label,
        max_attempts=retries,
    )
    return _extract_message_content(data)


def test_llm_connection(model_tier: str = "strong") -> None:
    """Optional quick test before running the full pipeline."""
    print(f"Testing {model_tier} model...")
    print("URL shape:", _mask_url_for_display(get_model_url(model_tier)))
    reply = azure_chat(
        [{"role": "user", "content": "Reply with exactly: OK"}],
        model_tier=model_tier,
        temperature=0,
        max_tokens=20,
        retries=2,
    )
    print("Model reply:", reply)


def _extract_json_object(text: str) -> str:
    """Extract the outermost JSON object from model output."""
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def parse_json_response(text: str) -> Dict[str, Any]:
    """Parse strict JSON, with fallback for fenced JSON or small leading/trailing commentary."""
    candidate = _extract_json_object(text)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError as exc:
        raise ValueError(
            "Model response was not valid JSON. Preview:\n"
            + text[:3000]
        ) from exc


print("Role-specific LLM helper functions ready")
print("Model config:", json.dumps(MODEL_CONFIG, indent=2))
print()
print("Before running the full pipeline, test both endpoints:")
print("test_llm_connection('strong')")
print("test_llm_connection('fast')")


Azure OpenAI full-URL configuration loaded
Strong endpoint (GPT-5.2): https://eyq***or.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gp***/chat/completions?api-version=...
Fast endpoint: https://eyq***or.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gp***/chat/completions?api-version=...
Role-specific LLM helper functions ready
Model config: {
  "fuzzy_evidence_mapper": "fast",
  "section_writer": "strong",
  "claims_register_builder": "strong",
  "ifrs_coverage_judge": "strong",
  "evidence_judge": "strong",
  "style_judge": "fast",
  "minimal_reviser": "strong",
  "whole_report_connectivity_judge": "strong"
}

Before running the full pipeline, test both endpoints:
test_llm_connection('strong')
test_llm_connection('fast')


In [3]:
# ============================================================
# CELL 3 — GENERAL UTILITIES
# ============================================================


def slugify(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        if default is not None:
            return default
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_text(path: Path, default: str = "") -> str:
    if not path.exists():
        return default
    return path.read_text(encoding="utf-8", errors="replace")


def write_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def flatten_json(obj: Any, prefix: str = "") -> Dict[str, Any]:
    """Flatten nested dict/list into path -> scalar/list/dict value."""
    out = {}

    if isinstance(obj, dict):
        for k, v in obj.items():
            new_prefix = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_json(v, new_prefix))
    elif isinstance(obj, list):
        if not obj:
            out[prefix] = []
        else:
            for i, v in enumerate(obj):
                new_prefix = f"{prefix}[{i}]"
                out.update(flatten_json(v, new_prefix))
    else:
        out[prefix] = obj

    return out


def get_by_path(obj: Any, path: str) -> Any:
    """Resolve paths like a.b[0].c."""
    if not path:
        return None
    cur = obj
    tokens = re.findall(r"([^\.\[\]]+)|(\[(\d+)\])", path)
    for name, _, idx in tokens:
        if name:
            if not isinstance(cur, dict) or name not in cur:
                return None
            cur = cur[name]
        elif idx:
            i = int(idx)
            if not isinstance(cur, list) or i >= len(cur):
                return None
            cur = cur[i]
    return cur


def value_preview(value: Any, limit: int = 260) -> str:
    text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit] + ("..." if len(text) > limit else "")


def is_empty_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, str) and not value.strip():
        return True
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False


def tokens(text: str) -> List[str]:
    return [t.lower() for t in re.findall(r"[A-Za-z][A-Za-z0-9_\-]+", str(text)) if len(t) > 2]


def normalize_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "mandatory"}


def now_id() -> str:
    return uuid.uuid4().hex[:10]

## Load inputs

The notebook is tolerant of different file structures. Preferred structure:

```text
/notebooks/gen_data/payloads/
  payload_BANK01_general_requirements.json
  payload_BANK01_governance.json
  payload_BANK01_strategy.json
  payload_BANK01_risk_management.json
  payload_BANK01_metrics_targets.json

/notebooks/gen_data/ifrs_requirements/
  general_requirements_requirements.json
  governance_requirements.json
  strategy_requirements.json
  risk_management_requirements.json
  metrics_and_targets_requirements.json

/notebooks/gen_data/style/style_system/
  authoring/
  judging/
  rendering/
```

In [4]:
# ============================================================
# CELL 4 — LOAD STYLE ARTIFACTS
# ============================================================

AUTHORING_DIR = STYLE_SYSTEM_DIR / "authoring"
JUDGING_DIR = STYLE_SYSTEM_DIR / "judging"
RENDERING_DIR = STYLE_SYSTEM_DIR / "rendering"

# Backward-compatible fallbacks if final organized folders are not present.
if not AUTHORING_DIR.exists():
    AUTHORING_DIR = STYLE_SYSTEM_DIR
if not JUDGING_DIR.exists():
    JUDGING_DIR = STYLE_SYSTEM_DIR / "judging"
if not RENDERING_DIR.exists():
    RENDERING_DIR = STYLE_SYSTEM_DIR / "rendering"

GLOBAL_STYLE = read_json(AUTHORING_DIR / "global_style_guide.json", default={})
STYLE_RUBRIC = read_json(JUDGING_DIR / "style_compliance_rubric.json", default={})

NO_COPYING_RULES = read_text(
    AUTHORING_DIR / "language_rules" / "no_copying_rules.md",
    default=read_text(STYLE_SYSTEM_DIR / "language_rules" / "no_copying_rules.md", default="")
)

TABLE_PATTERNS = read_json(
    AUTHORING_DIR / "table_patterns" / "table_patterns.json",
    default=read_json(STYLE_SYSTEM_DIR / "table_patterns" / "table_patterns.json", default={})
)

FORBIDDEN_TERMS = read_json(
    AUTHORING_DIR / "language_rules" / "forbidden_reference_terms.json",
    default=read_json(STYLE_SYSTEM_DIR / "language_rules" / "forbidden_reference_terms.json", default=[])
)

# Hardcoded safety fallback in case forbidden_reference_terms.json is absent.
FORBIDDEN_TERMS = sorted(set(FORBIDDEN_TERMS + [
    "Emirates NBD", "Emirates NBD Group", "DenizBank", "Emirates Islamic",
    "Dubai", "UAE", "AED", "CBUAE", "Sustainalytics", "KPMG",
    "Microsoft Sustainability Manager"
]))


def load_section_style(section_name: str) -> Dict[str, Any]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        AUTHORING_DIR / "section_style_guides" / f"{slug}_style.json",
        AUTHORING_DIR / "section_style_guides" / f"{slug}.json",
        STYLE_SYSTEM_DIR / "section_style_guides" / f"{slug}_style.json",
        STYLE_SYSTEM_DIR / "section_style_guides" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return read_json(path, default={})
    return {}


def load_section_blueprint(section_name: str) -> Dict[str, Any]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        AUTHORING_DIR / "section_blueprints" / f"{slug}_blueprint.json",
        AUTHORING_DIR / "section_blueprints" / f"{slug}.json",
        STYLE_SYSTEM_DIR / "section_blueprints" / f"{slug}_blueprint.json",
        STYLE_SYSTEM_DIR / "section_blueprints" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return read_json(path, default={})
    return {}

print("Loaded global style:", bool(GLOBAL_STYLE))
print("Loaded table patterns:", bool(TABLE_PATTERNS))
print("Loaded no-copying rules:", bool(NO_COPYING_RULES))
print("Forbidden terms count:", len(FORBIDDEN_TERMS))

Loaded global style: True
Loaded table patterns: True
Loaded no-copying rules: True
Forbidden terms count: 31


In [5]:
# ============================================================
# CELL 5 — LOAD REQUIREMENTS
# PATCHED: robust to section JSON files saved as:
# - list[dict]
# - {"requirements": list[dict]}
# - {"requirements": {requirement_id: dict/text}}
# - {requirement_id: dict/text}
# - {section_name: list[dict]}
# ============================================================


def find_requirements_file(section_name: str) -> Optional[Path]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        REQUIREMENTS_DIR / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / f"{slug}.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / "json" / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / "json" / f"{slug}.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def find_combined_requirements_file() -> Optional[Path]:
    candidates = [
        REQUIREMENTS_DIR / "ifrs_s1_s2_generation_requirements.json",
        REQUIREMENTS_DIR / "generation_requirements.json",
        REQUIREMENTS_DIR / "ifrs_s1_s2_requirements_kb_final.json",
        REQUIREMENTS_DIR / "requirements.json",
        GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_generation_requirements.json",
        GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_requirements_kb_final.json",
        GEN_DATA_DIR / "ifrs_s1_s2_generation_requirements.json",
        GEN_DATA_DIR / "ifrs_s1_s2_requirements_kb_final.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def _norm_key(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def _looks_like_requirement_dict(obj: Dict[str, Any]) -> bool:
    keys = set(obj.keys())
    return bool(keys.intersection({
        "requirement_id",
        "clean_requirement_text",
        "requirement_text",
        "source_paragraph_text",
        "paragraph_id",
        "report_section",
        "clause_path",
    }))


def rows_from_requirements_object(obj: Any, section_name: str = "") -> List[Any]:
    """
    Convert many possible JSON shapes into a list of requirement rows.

    This fixes the error:
        AttributeError: 'str' object has no attribute 'get'

    That error happens when a requirements JSON is a dict and Python iterates over
    its keys as strings instead of over requirement row dictionaries.
    """
    # Already a list.
    if isinstance(obj, list):
        return obj

    # Single string requirement.
    if isinstance(obj, str):
        return [obj]

    if not isinstance(obj, dict):
        return []

    # Single requirement object.
    if _looks_like_requirement_dict(obj):
        return [obj]

    # Common containers.
    for key in ["requirements", "rows", "records", "data", "items"]:
        if key in obj:
            nested = obj[key]
            # requirements can be a list or a dict keyed by requirement_id.
            if isinstance(nested, list):
                return nested
            if isinstance(nested, dict):
                # If nested is a single requirement object, wrap it.
                if _looks_like_requirement_dict(nested):
                    return [nested]
                rows = []
                for k, v in nested.items():
                    if isinstance(v, dict):
                        row = dict(v)
                        row.setdefault("requirement_id", k)
                        rows.append(row)
                    elif isinstance(v, str):
                        rows.append({"requirement_id": k, "requirement_text": v})
                    elif isinstance(v, list):
                        rows.extend(v)
                return rows
            if isinstance(nested, str):
                return [nested]

    # Dict keyed by section names.
    section_keys = {
        _norm_key(section_name),
        _norm_key(SECTION_SLUGS.get(section_name, "")),
        _norm_key(section_name.replace("and", "&")),
    }
    for k, v in obj.items():
        if _norm_key(k) in section_keys:
            return rows_from_requirements_object(v, section_name)

    # Dict keyed by requirement IDs.
    rows = []
    for k, v in obj.items():
        if isinstance(v, dict):
            row = dict(v)
            row.setdefault("requirement_id", k)
            rows.append(row)
        elif isinstance(v, str):
            rows.append({"requirement_id": k, "requirement_text": v})
        elif isinstance(v, list):
            # Could be {"General Requirements": [...]} but key did not match;
            # keep rows that look like requirements and filter later.
            rows.extend(v)

    return rows


def normalize_requirement(row: Any, section_name: str) -> Dict[str, Any]:
    # Accept strings safely.
    if isinstance(row, str):
        row = {
            "requirement_text": row,
            "requirement_id": now_id(),
            "report_section": section_name,
        }

    # Accept non-dict rows safely.
    if not isinstance(row, dict):
        row = {
            "requirement_text": str(row),
            "requirement_id": now_id(),
            "report_section": section_name,
        }

    requirement_text = (
        row.get("clean_requirement_text")
        or row.get("requirement_text")
        or row.get("source_paragraph_text")
        or row.get("text")
        or row.get("paragraph_text")
        or ""
    )

    # If evidence_tags is stored as a JSON string, normalize it to a list.
    evidence_tags = row.get("evidence_tags", [])
    if isinstance(evidence_tags, str):
        try:
            parsed = json.loads(evidence_tags)
            evidence_tags = parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            evidence_tags = [x.strip() for x in re.split(r"[,;|]", evidence_tags) if x.strip()]

    return {
        "requirement_id": str(row.get("requirement_id") or row.get("id") or now_id()),
        "standard": row.get("standard", ""),
        "paragraph_id": row.get("paragraph_id", row.get("paragraph", "")),
        "page": row.get("page", ""),
        "report_section": row.get("report_section", section_name),
        "requirement_text": str(requirement_text).strip(),
        "clause_path": row.get("clause_path", ""),
        "obligation_type": row.get("obligation_type", ""),
        "mandatory": normalize_bool(row.get("mandatory", True)),
        "evidence_tags": evidence_tags,
        "banking_relevance": row.get("banking_relevance", ""),
        "raw": row,
    }


def section_matches(row: Dict[str, Any], section_name: str) -> bool:
    sec = str(row.get("report_section", "")).strip()
    if not sec:
        return True  # section-specific files may not repeat the section name
    return _norm_key(sec) == _norm_key(section_name)


def load_requirements_for_section(section_name: str) -> List[Dict[str, Any]]:
    section_file = find_requirements_file(section_name)

    if section_file:
        obj = read_json(section_file)
        raw_rows = rows_from_requirements_object(obj, section_name)
        rows = [
            normalize_requirement(r, section_name)
            for r in raw_rows
        ]
        rows = [r for r in rows if r["requirement_text"] and section_matches(r, section_name)]
        print(f"Loaded requirements for {section_name} from section file: {section_file}")
        return rows

    combined_file = find_combined_requirements_file()
    if not combined_file:
        raise FileNotFoundError(
            "Could not find IFRS requirements. Place section JSON files in REQUIREMENTS_DIR "
            "or set IFRS_REQUIREMENTS_DIR in .env."
        )

    obj = read_json(combined_file)
    raw_rows = rows_from_requirements_object(obj, section_name)
    rows = []
    for r in raw_rows:
        nr = normalize_requirement(r, section_name)
        sec = str(nr.get("report_section", "")).strip()
        if _norm_key(sec) == _norm_key(section_name):
            rows.append(nr)

    print(f"Loaded requirements for {section_name} from combined file: {combined_file}")
    return rows


requirements_by_section = {
    section: load_requirements_for_section(section)
    for section in SECTIONS
}

for section, reqs in requirements_by_section.items():
    print(f"{section}: {len(reqs)} requirements")
    if len(reqs) == 0:
        print(f"  WARNING: no requirements found for {section}. Check report_section names or section JSON files.")


Loaded requirements for General Requirements from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\general_requirements_requirements.json
Loaded requirements for Governance from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\governance_requirements.json
Loaded requirements for Strategy from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\strategy_requirements.json
Loaded requirements for Risk Management from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\risk_management_requirements.json
Loaded requirements for Metrics and Targets fr

In [6]:
# ============================================================
# CELL 6 — LOAD PAYLOADS
# PATCHED: also searches /notebooks/BANK01 and common project folders.
# ============================================================


def payload_search_dirs() -> List[Path]:
    candidates = [
        PAYLOAD_DIR,
        NOTEBOOK_DIR / "payloads",
        NOTEBOOK_DIR / "data",
        GEN_DATA_DIR / "payloads",
        GEN_DATA_DIR / "BANK01",
        GEN_DATA_DIR / "data",
        CURRENT_DIR / "BANK01",
        CURRENT_DIR / "data",
    ]

    # Keep unique existing-or-configured paths in order.
    out = []
    seen = set()
    for p in candidates:
        p = Path(p).resolve()
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out


def find_payload_file(section_name: str) -> Optional[Path]:
    slug = SECTION_SLUGS[section_name]
    aliases = {
        "metrics_and_targets": ["metrics_targets", "metrics_and_targets", "metrics_targets"],
        "risk_management": ["risk_management", "risk"],
        "general_requirements": ["general_requirements", "general"],
        "governance": ["governance"],
        "strategy": ["strategy"],
    }[slug]

    filename_patterns = []
    for alias in aliases:
        filename_patterns.extend([
            f"payload_BANK01_{alias}.json",
            f"payload_BANK01_{alias}*.json",
            f"BANK01_{alias}.json",
            f"*{alias}*.json",
            f"{alias}.json",
        ])

    for folder in payload_search_dirs():
        if not folder.exists():
            continue
        for pattern in filename_patterns:
            matches = sorted(folder.glob(pattern))
            if matches:
                return matches[0]

    return None


def find_combined_payload_file() -> Optional[Path]:
    filename_patterns = [
        "payload_BANK01.json",
        "payload_BANK01*.json",
        "BANK01.json",
        "payload.json",
        "*payload*.json",
    ]

    for folder in payload_search_dirs():
        if not folder.exists():
            continue
        for pattern in filename_patterns:
            matches = sorted(folder.glob(pattern))
            if matches:
                # Avoid selecting section payload if a combined one exists later.
                section_hint = matches[0].name.lower()
                if any(x in section_hint for x in ["governance", "strategy", "risk_management", "metrics", "general_requirements"]):
                    continue
                return matches[0]

    return None


def load_payload_for_section(section_name: str) -> Dict[str, Any]:
    section_file = find_payload_file(section_name)
    if section_file:
        print(f"Loaded payload for {section_name} from section file: {section_file}")
        return read_json(section_file)

    combined_file = find_combined_payload_file()
    if combined_file:
        print(f"Loaded payload for {section_name} from combined file: {combined_file}")
        combined = read_json(combined_file)
        slug = SECTION_SLUGS[section_name]
        possible_keys = [
            slug,
            slug.replace("metrics_and_targets", "metrics_targets"),
            section_name,
            section_name.lower(),
            section_name.replace(" ", "_").lower(),
        ]
        for key in possible_keys:
            if isinstance(combined, dict) and key in combined:
                return combined[key]
        return combined

    searched = "\n".join([f"- {p}" for p in payload_search_dirs()])
    raise FileNotFoundError(
        "Could not find payload files.\n\n"
        "Searched these folders:\n"
        f"{searched}\n\n"
        "Expected examples:\n"
        "- payload_BANK01_governance.json\n"
        "- payload_BANK01_strategy.json\n"
        "- payload_BANK01_risk_management.json\n"
        "- payload_BANK01_metrics_targets.json\n"
        "- payload_BANK01_general_requirements.json\n"
        "- payload_BANK01.json"
    )


payloads_by_section = {
    section: load_payload_for_section(section)
    for section in SECTIONS
}

for section, payload in payloads_by_section.items():
    flat_count = len(flatten_json(payload))
    print(f"{section}: payload fields={flat_count}")


Loaded payload for General Requirements from combined file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01.json
Loaded payload for Governance from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_governance.json
Loaded payload for Strategy from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_strategy.json
Loaded payload for Risk Management from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_risk_management.json
Loaded payload for Metrics and Targets from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_metrics_targets.json
General Requirements: payload fields=5122
Governance: payload fields=658
Strategy: payload fields=1098
Risk Management: payload fields=3265
Metrics and Targets: payload fields=1028


## Deterministic evidence mapping and coverage

The evidence mapper is code-first. It maps requirements to actual payload paths. The optional LLM fuzzy mapper can suggest matches, but the path must still resolve to the payload.

Missing requirements are written to JSON audit outputs and are **not passed to the Writer as report content**.

In [7]:
# ============================================================
# CELL 7 — DETERMINISTIC EVIDENCE MAPPER
# ============================================================

STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "into", "about", "their",
    "shall", "should", "must", "entity", "entities", "information", "disclose",
    "disclosure", "related", "sustainability", "climate", "risks", "risk", "opportunities",
    "opportunity", "reporting", "period", "including", "describe", "explain",
}

TAG_SYNONYMS = {
    "governance": ["board", "committee", "oversight", "responsibility", "role", "governance"],
    "strategy": ["strategy", "business_model", "value_chain", "time_horizon", "financial_effects"],
    "risk_management": ["risk", "identify", "assess", "manage", "monitor", "control", "process"],
    "metrics": ["metric", "target", "value", "unit", "emissions", "scope", "baseline"],
    "targets": ["target", "baseline", "progress", "metric", "goal"],
    "financial_effects": ["financial", "cash", "position", "performance", "flows", "cost", "revenue"],
    "value_chain": ["upstream", "downstream", "operations", "supplier", "customer", "portfolio"],
    "materiality": ["material", "materiality", "assessment", "topic", "impact"],
}


def requirement_keywords(req: Dict[str, Any]) -> List[str]:
    parts = [
        req.get("requirement_text", ""),
        req.get("clause_path", ""),
        req.get("obligation_type", ""),
        req.get("banking_relevance", ""),
    ]
    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        try:
            tags = json.loads(tags)
        except Exception:
            tags = re.split(r"[,;|]", tags)
    parts.extend([str(t) for t in tags])

    kws = []
    for part in parts:
        kws.extend(tokens(part))
    expanded = []
    for kw in kws:
        expanded.append(kw)
        expanded.extend(TAG_SYNONYMS.get(kw, []))
    return sorted(set([k for k in expanded if k not in STOPWORDS]))


def field_keywords(path: str, value: Any) -> List[str]:
    text = path.replace("_", " ").replace(".", " ")
    if isinstance(value, (str, int, float, bool)):
        text += " " + str(value)
    elif isinstance(value, (dict, list)):
        text += " " + value_preview(value, limit=500)
    return [t for t in tokens(text) if t not in STOPWORDS]


def evidence_score(req_kws: List[str], path: str, value: Any) -> Tuple[int, List[str]]:
    f_kws = set(field_keywords(path, value))
    r_kws = set(req_kws)
    overlap = sorted(r_kws.intersection(f_kws))
    score = len(overlap)

    # Boost exact-ish field path matches.
    path_l = path.lower()
    for kw in r_kws:
        if kw in path_l:
            score += 1

    return score, overlap


def build_evidence_map_for_section(
    section_name: str,
    requirements: List[Dict[str, Any]],
    payload: Dict[str, Any],
    top_k: int = 8,
    min_score: int = 2,
) -> List[Dict[str, Any]]:
    flat = flatten_json(payload)
    non_empty_items = [(p, v) for p, v in flat.items() if not is_empty_value(v)]
    mapped = []

    for req in requirements:
        req_kws = requirement_keywords(req)
        candidates = []
        for path, value in non_empty_items:
            score, overlap = evidence_score(req_kws, path, value)
            if score >= min_score:
                candidates.append({
                    "payload_path": path,
                    "value_preview": value_preview(value),
                    "value_type": type(value).__name__,
                    "match_score": score,
                    "matched_keywords": overlap,
                })
        candidates = sorted(candidates, key=lambda x: x["match_score"], reverse=True)[:top_k]
        mapped.append({
            "requirement_id": req["requirement_id"],
            "section_name": section_name,
            "requirement_text": req["requirement_text"],
            "mandatory": req["mandatory"],
            "evidence_candidates": candidates,
            "mapping_method": "deterministic_lexical",
        })
    return mapped

# Build and save evidence maps.
evidence_maps_by_section = {}
for section in SECTIONS:
    evidence_map = build_evidence_map_for_section(
        section,
        requirements_by_section[section],
        payloads_by_section[section],
    )
    evidence_maps_by_section[section] = evidence_map
    path = DIRS["evidence_maps"] / f"evidence_map_{SECTION_SLUGS[section]}.json"
    write_json(evidence_map, path)
    print(section, "mapped", len(evidence_map), "requirements ->", path)

General Requirements mapped 3 requirements -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_general_requirements.json
Governance mapped 3 requirements -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_governance.json
Strategy mapped 3 requirements -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_strategy.json
Risk Management mapped 3 requirements -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_risk_management.json
Metrics and Targets mapped 3 requirements -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_metrics_and_targets.j

In [8]:
# ============================================================
# CELL 8 — OPTIONAL FUZZY EVIDENCE MAPPER LLM FALLBACK
# Use only for unresolved requirements. The path still must exist.
# ============================================================


def fuzzy_map_unresolved_requirements(
    section_name: str,
    requirements: List[Dict[str, Any]],
    payload: Dict[str, Any],
    evidence_map: List[Dict[str, Any]],
    max_unresolved: int = 20,
) -> List[Dict[str, Any]]:
    unresolved = [m for m in evidence_map if not m["evidence_candidates"]]
    if not unresolved or not USE_FUZZY_EVIDENCE_MAPPER:
        return evidence_map

    unresolved = unresolved[:max_unresolved]
    flat = flatten_json(payload)
    payload_catalog = [
        {"payload_path": p, "value_preview": value_preview(v, 120)}
        for p, v in flat.items()
        if not is_empty_value(v)
    ][:300]

    prompt = {
        "section_name": section_name,
        "task": "Suggest payload paths that may support unresolved IFRS requirements. Only use paths from payload_catalog.",
        "rules": [
            "Do not invent payload paths.",
            "Return an empty list if no path supports a requirement.",
            "A suggested path must be semantically relevant, not merely same section.",
        ],
        "unresolved_requirements": [
            {
                "requirement_id": m["requirement_id"],
                "requirement_text": m["requirement_text"],
                "mandatory": m["mandatory"],
            }
            for m in unresolved
        ],
        "payload_catalog": payload_catalog,
    }

    messages = [
        {"role": "system", "content": "You are a precise evidence mapping assistant. Return JSON only."},
        {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)},
    ]
    raw = azure_chat(
        messages,
        model_tier=MODEL_CONFIG["fuzzy_evidence_mapper"],
        temperature=0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw)
    suggestions = obj.get("suggestions", [])

    existing_paths = set(flat.keys())
    by_req = defaultdict(list)
    for s in suggestions:
        rid = s.get("requirement_id")
        for path in s.get("payload_paths", []):
            if path in existing_paths:
                by_req[rid].append({
                    "payload_path": path,
                    "value_preview": value_preview(flat[path]),
                    "value_type": type(flat[path]).__name__,
                    "match_score": int(s.get("confidence", 1)),
                    "matched_keywords": ["llm_fuzzy_match"],
                    "llm_reason": s.get("reason", ""),
                })

    for m in evidence_map:
        if not m["evidence_candidates"] and m["requirement_id"] in by_req:
            m["evidence_candidates"] = by_req[m["requirement_id"]]
            m["mapping_method"] = "llm_fuzzy_verified_path"

    return evidence_map

if USE_FUZZY_EVIDENCE_MAPPER:
    for section in SECTIONS:
        updated = fuzzy_map_unresolved_requirements(
            section,
            requirements_by_section[section],
            payloads_by_section[section],
            evidence_maps_by_section[section],
        )
        evidence_maps_by_section[section] = updated
        write_json(updated, DIRS["evidence_maps"] / f"evidence_map_{SECTION_SLUGS[section]}.json")
        print("Fuzzy mapping completed for", section)
else:
    print("Fuzzy evidence mapper disabled.")

Fuzzy evidence mapper disabled.


In [9]:
# ============================================================
# CELL 9 — COVERAGE CLASSIFIER + MISSING REQUIREMENTS REGISTER
# Missing requirements are NOT report content. They are audit JSON.
# ============================================================


def classify_requirement_coverage(req: Dict[str, Any], mapping: Dict[str, Any]) -> Dict[str, Any]:
    candidates = mapping.get("evidence_candidates", [])
    mandatory = req.get("mandatory", True)

    if candidates:
        # High-confidence coverage if at least one strong lexical/fuzzy match exists.
        max_score = max([c.get("match_score", 0) for c in candidates] or [0])
        status = "covered" if max_score >= 3 else "partially_covered"
    else:
        if mandatory:
            status = "not_available_in_payload"
        else:
            # Restricted: non-mandatory missing items are not automatically N/A;
            # still logged as missing unless a condition explicitly justifies N/A.
            raw_text = json.dumps(req.get("raw", {}), ensure_ascii=False).lower()
            if any(x in raw_text for x in ["if applicable", "when applicable", "where applicable", "conditional"]):
                status = "not_applicable"
            else:
                status = "not_available_in_payload"

    return {
        "requirement_id": req["requirement_id"],
        "standard": req.get("standard", ""),
        "paragraph_id": req.get("paragraph_id", ""),
        "report_section": req.get("report_section", ""),
        "requirement_text": req.get("requirement_text", ""),
        "mandatory": mandatory,
        "coverage_status": status,
        "evidence_count": len(candidates),
        "evidence_paths": [c["payload_path"] for c in candidates],
        "not_applicable_justification": "Conditional/non-mandatory requirement with no relevant synthetic payload evidence." if status == "not_applicable" else "",
    }


def build_coverage_and_missing_register(section_name: str) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    reqs = requirements_by_section[section_name]
    maps = {m["requirement_id"]: m for m in evidence_maps_by_section[section_name]}

    coverage = []
    missing = []

    for req in reqs:
        cov = classify_requirement_coverage(req, maps[req["requirement_id"]])
        coverage.append(cov)

        if cov["coverage_status"] == "not_available_in_payload":
            missing.append({
                "requirement_id": cov["requirement_id"],
                "standard": cov["standard"],
                "paragraph_id": cov["paragraph_id"],
                "report_section": section_name,
                "mandatory": cov["mandatory"],
                "requirement_text": cov["requirement_text"],
                "coverage_status": "not_available_in_payload",
                "reason": "No sufficiently relevant payload evidence was identified for this requirement.",
                "action_needed": "Add evidence for this requirement to the section payload or map an existing payload field manually.",
                "report_instruction": "Do not mention this missing requirement in the generated report.",
            })

    missing_register = {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "policy": "Missing requirements are recorded here and are excluded from report prose.",
        "missing_requirements": missing,
    }
    return coverage, missing_register

coverage_by_section = {}
missing_registers_by_section = {}

for section in SECTIONS:
    coverage, missing_register = build_coverage_and_missing_register(section)
    coverage_by_section[section] = coverage
    missing_registers_by_section[section] = missing_register

    slug = SECTION_SLUGS[section]
    write_json(coverage, DIRS["coverage"] / f"coverage_matrix_{slug}.json")
    write_json(missing_register, DIRS["missing_requirements"] / f"missing_requirements_{slug}.json")

    counts = Counter([c["coverage_status"] for c in coverage])
    print(section, dict(counts), "missing:", len(missing_register["missing_requirements"]))

combined_missing = {
    "pipeline_mode": PIPELINE_MODE,
    "policy": "The report contains only evidence-supported disclosures. Missing requirements are stored here, not in the report.",
    "sections": missing_registers_by_section,
}
write_json(combined_missing, DIRS["missing_requirements"] / "missing_requirements_all_sections.json")

General Requirements {'not_available_in_payload': 2, 'partially_covered': 1} missing: 2
Governance {'covered': 2, 'partially_covered': 1} missing: 0
Strategy {'not_available_in_payload': 2, 'partially_covered': 1} missing: 2
Risk Management {'not_available_in_payload': 1, 'partially_covered': 2} missing: 1
Metrics and Targets {'not_available_in_payload': 2, 'covered': 1} missing: 2


## Deterministic section planner

The planner is code-first. It builds a disclosure plan from covered requirements, available evidence, section blueprints, and table patterns. Missing requirements are excluded from the plan and kept only in JSON audit files.

In [10]:
# ============================================================
# CELL 10 — DETERMINISTIC SECTION PLANNER
# ============================================================

DEFAULT_SECTION_SUBSECTIONS = {
    "General Requirements": [
        {"heading": "Basis of preparation", "keywords": ["basis", "preparation", "compliance", "standard"]},
        {"heading": "Reporting boundary and connected information", "keywords": ["boundary", "entity", "connected", "financial"]},
        {"heading": "Materiality and judgement", "keywords": ["material", "judgement", "estimate", "assumption"]},
    ],
    "Governance": [
        {"heading": "Governance oversight", "keywords": ["board", "committee", "oversight", "governance"]},
        {"heading": "Roles, responsibilities and escalation", "keywords": ["responsibility", "role", "management", "escalation", "report"]},
        {"heading": "Skills, controls and monitoring", "keywords": ["skill", "competence", "control", "monitor", "training"]},
    ],
    "Strategy": [
        {"heading": "Business model and value chain", "keywords": ["business", "model", "value", "chain", "upstream", "downstream"]},
        {"heading": "Sustainability-related risks and opportunities", "keywords": ["risk", "opportunity", "material", "impact"]},
        {"heading": "Time horizons and financial effects", "keywords": ["time", "horizon", "financial", "cash", "performance"]},
        {"heading": "Resilience and strategic response", "keywords": ["resilience", "strategy", "response", "scenario"]},
    ],
    "Risk Management": [
        {"heading": "Risk identification and assessment", "keywords": ["identify", "assessment", "assess", "risk"]},
        {"heading": "Risk management processes and controls", "keywords": ["manage", "process", "control", "mitigation"]},
        {"heading": "Monitoring, reporting and integration", "keywords": ["monitor", "report", "integrat", "escalation"]},
    ],
    "Metrics and Targets": [
        {"heading": "Metrics register", "keywords": ["metric", "value", "unit", "measure"]},
        {"heading": "Targets and progress", "keywords": ["target", "baseline", "progress", "goal"]},
        {"heading": "Methodology and source traceability", "keywords": ["method", "source", "boundary", "definition"]},
    ],
}


def choose_subsection(section_name: str, requirement_text: str) -> str:
    req_tokens = set(tokens(requirement_text))
    candidates = DEFAULT_SECTION_SUBSECTIONS[section_name]
    scored = []
    for sub in candidates:
        score = sum(1 for kw in sub["keywords"] if any(kw in t for t in req_tokens))
        scored.append((score, sub["heading"]))
    scored.sort(reverse=True)
    return scored[0][1] if scored and scored[0][0] > 0 else candidates[0]["heading"]


def build_disclosure_plan(section_name: str) -> Dict[str, Any]:
    coverage = coverage_by_section[section_name]
    reqs_by_id = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    maps_by_id = {m["requirement_id"]: m for m in evidence_maps_by_section[section_name]}

    include_statuses = {"covered"}
    if ALLOW_PARTIAL_COVERAGE:
        include_statuses.add("partially_covered")

    supported = [c for c in coverage if c["coverage_status"] in include_statuses and c["evidence_count"] > 0]

    subsections = []
    subsection_map = defaultdict(lambda: {
        "heading": "",
        "purpose": "",
        "requirement_ids": [],
        "evidence_paths": [],
        "recommended_format": "short narrative",
    })

    for cov in supported:
        req = reqs_by_id[cov["requirement_id"]]
        heading = choose_subsection(section_name, req["requirement_text"])
        item = subsection_map[heading]
        item["heading"] = heading
        item["purpose"] = f"Address evidence-supported {section_name.lower()} disclosure requirements related to {heading.lower()}."
        item["requirement_ids"].append(cov["requirement_id"])
        item["evidence_paths"].extend(cov["evidence_paths"])

    for heading, item in subsection_map.items():
        item["requirement_ids"] = sorted(set(item["requirement_ids"]))
        item["evidence_paths"] = sorted(set(item["evidence_paths"]))
        if section_name == "Metrics and Targets":
            item["recommended_format"] = "table-first with brief narrative"
        elif section_name in {"Governance", "Risk Management"}:
            item["recommended_format"] = "narrative plus responsibility/process table if evidence supports it"
        elif section_name == "Strategy":
            item["recommended_format"] = "structured narrative plus value-chain/time-horizon table if evidence supports it"
        subsections.append(dict(item))

    if not subsections:
        subsections = [{
            "heading": section_name,
            "purpose": "No evidence-supported requirements were available for report drafting.",
            "requirement_ids": [],
            "evidence_paths": [],
            "recommended_format": "omit section content or mark for human review",
        }]

    # Recommended tables from style table patterns.
    recommended_tables = []
    recommended_columns = TABLE_PATTERNS.get("recommended_columns_by_table_type", {}) if isinstance(TABLE_PATTERNS, dict) else {}
    for table_name, cols in recommended_columns.items():
        t = table_name.lower()
        if section_name.lower().split()[0] in t or (
            section_name == "Metrics and Targets" and "metrics" in t
        ) or (
            section_name == "Risk Management" and "risk" in t
        ):
            recommended_tables.append({"table_name": table_name, "columns": cols})

    plan = {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "policy": "Plan includes only covered/partially covered evidence-supported requirements. Missing requirements are excluded from report content.",
        "subsections": subsections,
        "recommended_tables": recommended_tables[:4],
        "excluded_missing_requirement_count": len(missing_registers_by_section[section_name]["missing_requirements"]),
        "section_blueprint": load_section_blueprint(section_name),
    }
    return plan

plans_by_section = {}
for section in SECTIONS:
    plan = build_disclosure_plan(section)
    plans_by_section[section] = plan
    write_json(plan, DIRS["plans"] / f"disclosure_plan_{SECTION_SLUGS[section]}.json")
    print(section, "subsections:", len(plan["subsections"]), "recommended tables:", len(plan["recommended_tables"]))

General Requirements subsections: 1 recommended tables: 1
Governance subsections: 1 recommended tables: 2
Strategy subsections: 1 recommended tables: 1
Risk Management subsections: 2 recommended tables: 3
Metrics and Targets subsections: 1 recommended tables: 1


## LLM writer and claims builder

The writer only receives supported requirements and supported evidence. It must not mention missing requirements, synthetic data, missing payloads, or unavailable information.

In [11]:
# ============================================================
# CELL 11 — CONTEXT PACKER FOR LLM AGENTS
# ============================================================


def requirement_subset(section_name: str, requirement_ids: List[str]) -> List[Dict[str, Any]]:
    reqs = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    return [reqs[rid] for rid in requirement_ids if rid in reqs]


def evidence_subset(section_name: str, evidence_paths: List[str], limit_value_chars: int = 500) -> List[Dict[str, Any]]:
    payload = payloads_by_section[section_name]
    out = []
    for path in sorted(set(evidence_paths)):
        value = get_by_path(payload, path)
        if value is not None:
            out.append({
                "payload_path": path,
                "value_preview": value_preview(value, limit=limit_value_chars),
                "value_type": type(value).__name__,
            })
    return out


def build_writer_context(section_name: str) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    req_ids = sorted(set([rid for sub in plan["subsections"] for rid in sub.get("requirement_ids", [])]))
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))

    return {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "critical_rules": [
            "Write only evidence-supported disclosures.",
            "Do not mention missing requirements, missing payload fields, unavailable synthetic data, or data gaps in the report prose.",
            "Do not invent committees, policies, tools, targets, metrics, dates, currencies, financial effects, or maturity claims.",
            "Do not use the PDF layout guide or emit layout placeholders.",
            "Use the target payload only as factual evidence; style guides affect wording only.",
        ],
        "supported_requirements": requirement_subset(section_name, req_ids),
        "disclosure_plan": plan,
        "evidence_items": evidence_subset(section_name, ev_paths),
        "authoring_style": GLOBAL_STYLE,
        "section_style": load_section_style(section_name),
        "section_blueprint": load_section_blueprint(section_name),
        "table_patterns": TABLE_PATTERNS,
        "no_copying_rules": NO_COPYING_RULES,
    }


def truncate_context(obj: Any, max_chars: int = 60000) -> str:
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...TRUNCATED_FOR_TOKEN_LIMIT..."

In [12]:
# ============================================================
# CELL 12 — SECTION WRITER AGENT
# ============================================================


def write_section_draft(section_name: str) -> Dict[str, Any]:
    context = build_writer_context(section_name)

    system = """
You are an IFRS S1/S2 sustainability disclosure writer.
You write audit-ready Markdown sections using only the provided evidence.
You must not invent facts. You must not mention missing payload fields or synthetic-data limitations in report prose.
Return JSON only.
""".strip()

    user = f"""
Write the {section_name} section in Markdown.

Rules:
1. Use only supported_requirements and evidence_items.
2. Do not disclose or mention requirements that are missing from the payload.
3. Do not write phrases such as "not available in the payload", "synthetic dataset", "data not provided", or "missing requirement".
4. Do not use PDF layout placeholders such as divider pages, image placeholders, or page spreads.
5. Use neutral, IFRS-aligned, non-promotional language.
6. Use tables only when evidence supports table content.
7. Target-company-specific names are allowed only if present in evidence_items.
8. Return valid JSON with keys: section_name, draft_markdown, writer_notes.

Context:
{truncate_context(context)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["section_writer"],
        temperature=0.15,
        max_tokens=6000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw)
    obj.setdefault("section_name", section_name)
    obj.setdefault("draft_markdown", "")
    return obj

In [13]:
# ============================================================
# CELL 13 — CLAIMS REGISTER BUILDER AGENT
# ============================================================


def build_claims_register(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    req_ids = sorted(set([rid for sub in plan["subsections"] for rid in sub.get("requirement_ids", [])]))
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))

    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "supported_requirements": requirement_subset(section_name, req_ids),
        "evidence_items": evidence_subset(section_name, ev_paths, limit_value_chars=900),
        "instructions": [
            "Extract every material factual claim from the draft.",
            "For each claim, list entities, numbers, dates, evidence_sources and requirement_ids.",
            "evidence_sources must be exact payload_path values from evidence_items.",
            "If a claim has no evidence source, mark supported=false and explain why.",
            "Do not create evidence paths that are not in evidence_items.",
        ],
    }

    system = "You are a strict audit claims-register builder. Return JSON only."
    user = f"""
Build a claims register for this generated report section.

Return JSON with keys:
- section_name
- claims: list of objects with claim_id, claim_text, claim_type, entities, numbers, dates, evidence_sources, requirement_ids, supported, support_notes

Context:
{truncate_context(context)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["claims_register_builder"],
        temperature=0,
        max_tokens=5000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw)
    obj.setdefault("section_name", section_name)
    obj.setdefault("claims", [])
    return obj

## Deterministic gates

These gates run before LLM judges and after every revision:

1. Claims integrity gate.
2. Factlock number/entity gate.
3. Reference firewall gate.
4. Report cleanliness gate.

If deterministic gates fail, the pipeline revises or escalates without wasting judge calls.

In [14]:
# ============================================================
# CELL 14 — DETERMINISTIC GATES
# ============================================================

NUMBER_PATTERN = re.compile(
    r"(?<![A-Za-z0-9])(?:\d{1,3}(?:[, ]\d{3})+|\d+)(?:\.\d+)?\s?(?:%|bps|AED|USD|EUR|tCO2e|tonnes|years?|days?)?",
    flags=re.IGNORECASE,
)

ENTITY_PATTERN = re.compile(
    r"\b(?:[A-Z][A-Za-z0-9&\-/]+(?:\s+[A-Z][A-Za-z0-9&\-/]+){1,6})\b"
)

REPORT_CLEANLINESS_BLOCKLIST = [
    "synthetic dataset",
    "synthetic data",
    "synthetic payload",
    "missing from the payload",
    "not available in the payload",
    "not included in the payload",
    "payload does not include",
    "data not provided",
    "missing requirement",
    "not available for this reporting cycle",
]


def extract_numbers(text: str) -> List[str]:
    return sorted(set([m.group(0).strip() for m in NUMBER_PATTERN.finditer(text)]))


def extract_entities(text: str) -> List[str]:
    raw = [m.group(0).strip() for m in ENTITY_PATTERN.finditer(text)]
    ignore = {"IFRS", "General Requirements", "Risk Management", "Metrics and Targets"}
    return sorted(set([x for x in raw if x not in ignore and not x.startswith("Table ") and not x.startswith("Figure ")]))


def payload_text(section_name: str) -> str:
    return json.dumps(payloads_by_section[section_name], ensure_ascii=False)


def claims_integrity_gate(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    payload = payloads_by_section[section_name]
    req_ids = {r["requirement_id"] for r in requirements_by_section[section_name]}
    failures = []

    claims = claims_register.get("claims", [])
    for claim in claims:
        cid = claim.get("claim_id", "UNKNOWN")
        if claim.get("supported") is False:
            failures.append({"claim_id": cid, "issue": "claim_marked_unsupported", "claim": claim.get("claim_text", "")})

        evidence_sources = claim.get("evidence_sources", [])
        if not evidence_sources:
            failures.append({"claim_id": cid, "issue": "no_evidence_source", "claim": claim.get("claim_text", "")})
        for src in evidence_sources:
            if get_by_path(payload, src) is None:
                failures.append({"claim_id": cid, "issue": "evidence_source_does_not_resolve", "evidence_source": src})

        for rid in claim.get("requirement_ids", []):
            if rid not in req_ids:
                failures.append({"claim_id": cid, "issue": "unknown_requirement_id", "requirement_id": rid})

    return {
        "gate_name": "claims_integrity",
        "passed": len(failures) == 0,
        "failures": failures,
        "claim_count": len(claims),
    }


def factlock_gate(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    draft_numbers = extract_numbers(draft_markdown)
    draft_entities = extract_entities(draft_markdown)

    claim_numbers = set()
    claim_entities = set()
    for claim in claims_register.get("claims", []):
        claim_numbers.update([str(x).strip() for x in claim.get("numbers", [])])
        claim_entities.update([str(x).strip() for x in claim.get("entities", [])])

    p_text = payload_text(section_name).lower()
    failures = []

    for num in draft_numbers:
        n = num.lower().strip()
        if n not in p_text and num not in claim_numbers:
            # Table/section numbers are allowed.
            if not re.fullmatch(r"\d+(\.\d+)?", num.strip()):
                failures.append({"type": "number_not_in_payload_or_claims", "value": num})

    # We keep entity factlock conservative to avoid false positives from headings.
    for ent in draft_entities:
        ent_l = ent.lower()
        if ent_l in {"ifrs s1", "ifrs s2"}:
            continue
        if ent_l not in p_text and ent not in claim_entities:
            if not any(term.lower() == ent_l for term in ["Risk Management", "Metrics and Targets", "General Requirements"]):
                failures.append({"type": "entity_not_in_payload_or_claims", "value": ent})

    return {
        "gate_name": "factlock_numbers_entities",
        "passed": len(failures) == 0,
        "failures": failures[:100],
        "draft_numbers": draft_numbers,
        "draft_entities": draft_entities[:100],
    }


def reference_firewall_gate(draft_markdown: str) -> Dict[str, Any]:
    text_l = draft_markdown.lower()
    hits = [term for term in FORBIDDEN_TERMS if term and term.lower() in text_l]
    return {
        "gate_name": "reference_firewall",
        "passed": len(hits) == 0,
        "forbidden_term_hits": hits,
    }


def report_cleanliness_gate(draft_markdown: str) -> Dict[str, Any]:
    text_l = draft_markdown.lower()
    hits = [phrase for phrase in REPORT_CLEANLINESS_BLOCKLIST if phrase in text_l]
    return {
        "gate_name": "report_cleanliness_no_missing_payload_language",
        "passed": len(hits) == 0,
        "blocked_phrase_hits": hits,
    }


def run_deterministic_gates(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    gates = [
        claims_integrity_gate(section_name, claims_register),
        factlock_gate(section_name, draft_markdown, claims_register),
        reference_firewall_gate(draft_markdown),
        report_cleanliness_gate(draft_markdown),
    ]
    return {
        "section_name": section_name,
        "passed": all(g["passed"] for g in gates),
        "gates": gates,
    }

In [15]:
# ============================================================
# CELL 15 — LLM JUDGES
# ============================================================


def judge_ifrs_coverage(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "coverage_matrix": coverage_by_section[section_name],
        "missing_requirements_policy": "Missing requirements must be absent from report prose and present in missing_requirements.json.",
        "missing_requirements_register": missing_registers_by_section[section_name],
        "claims_register": claims_register,
    }
    system = "You are an IFRS S1/S2 coverage judge. Return JSON only."
    user = f"""
Judge the generated section against available IFRS requirements.

Important policy:
- Do NOT fail the section because requirements marked not_available_in_payload are absent from the report.
- Fail if a missing requirement is invented in the report.
- Fail if a covered requirement is not addressed despite available evidence.
- Missing requirements must be tracked in missing_requirements_register, not in report prose.

Return JSON with: approved, ifrs_coverage_score_0_to_10, missing_supported_requirements, invented_missing_requirements, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["ifrs_coverage_judge"],
        temperature=0,
        max_tokens=3500,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw)


def judge_evidence_support(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "claims_register": claims_register,
        "evidence_items": evidence_subset(section_name, ev_paths, limit_value_chars=900),
        "rules": [
            "Every material claim must be supported by payload evidence.",
            "No invented metrics, targets, committees, policies, tools, dates, currencies, or financial effects.",
            "Do not penalize omission of missing requirements listed in missing_requirements.json.",
        ],
    }
    system = "You are a strict evidence support judge. Return JSON only."
    user = f"""
Judge whether the section contains unsupported claims.

Return JSON with: approved, evidence_score_0_to_10, unsupported_claims, questionable_claims, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["evidence_judge"],
        temperature=0,
        max_tokens=3500,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw)


def judge_style(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "global_style_guide": GLOBAL_STYLE,
        "section_style": load_section_style(section_name),
        "style_compliance_rubric": STYLE_RUBRIC,
        "no_copying_rules": NO_COPYING_RULES,
    }
    system = "You are a sustainability report style judge. Return JSON only."
    user = f"""
Judge whether the section follows the approved authoring style.

Return JSON with: approved, style_score_0_to_10, voice_issues, structure_issues, wording_issues, table_figure_issues, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["style_judge"],
        temperature=0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw)


def run_llm_judges(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "section_name": section_name,
        "ifrs_coverage_judge": judge_ifrs_coverage(section_name, draft_markdown, claims_register),
        "evidence_judge": judge_evidence_support(section_name, draft_markdown, claims_register),
        "style_judge": judge_style(section_name, draft_markdown),
    }

In [16]:
# ============================================================
# CELL 16 — COMPOSITE APPROVAL GATE
# ============================================================

APPROVAL_THRESHOLDS = {
    "ifrs_coverage_score_min": float(os.getenv("IFRS_COVERAGE_SCORE_MIN", "8.0")),
    "evidence_score_min": float(os.getenv("EVIDENCE_SCORE_MIN", "8.0")),
    "style_score_min": float(os.getenv("STYLE_SCORE_MIN", "7.5")),
}


def _score(obj: Dict[str, Any], *names: str) -> float:
    for name in names:
        if name in obj:
            try:
                return float(obj[name])
            except Exception:
                pass
    return 0.0


def composite_approval_gate(
    section_name: str,
    deterministic_result: Dict[str, Any],
    judge_results: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    if not deterministic_result.get("passed", False):
        return {
            "section_name": section_name,
            "approved": False,
            "reason": "deterministic_gates_failed",
            "required_fixes": deterministic_result,
        }

    if judge_results is None:
        return {
            "section_name": section_name,
            "approved": False,
            "reason": "llm_judges_not_run",
            "required_fixes": [],
        }

    ifrs = judge_results.get("ifrs_coverage_judge", {})
    evidence = judge_results.get("evidence_judge", {})
    style = judge_results.get("style_judge", {})

    ifrs_score = _score(ifrs, "ifrs_coverage_score_0_to_10", "score")
    evidence_score = _score(evidence, "evidence_score_0_to_10", "score")
    style_score = _score(style, "style_score_0_to_10", "score")

    failures = []
    if ifrs_score < APPROVAL_THRESHOLDS["ifrs_coverage_score_min"] or not ifrs.get("approved", False):
        failures.append({"judge": "ifrs_coverage_judge", "score": ifrs_score, "required_fixes": ifrs.get("required_fixes", [])})
    if evidence_score < APPROVAL_THRESHOLDS["evidence_score_min"] or not evidence.get("approved", False):
        failures.append({"judge": "evidence_judge", "score": evidence_score, "required_fixes": evidence.get("required_fixes", [])})
    if style_score < APPROVAL_THRESHOLDS["style_score_min"] or not style.get("approved", False):
        failures.append({"judge": "style_judge", "score": style_score, "required_fixes": style.get("required_fixes", [])})

    return {
        "section_name": section_name,
        "approved": len(failures) == 0,
        "scores": {
            "ifrs_coverage": ifrs_score,
            "evidence": evidence_score,
            "style": style_score,
        },
        "failures": failures,
        "thresholds": APPROVAL_THRESHOLDS,
    }

In [17]:
# ============================================================
# CELL 17 — MINIMAL REVISER AGENT
# ============================================================


def collect_fix_instructions(deterministic_result: Dict[str, Any], judge_results: Optional[Dict[str, Any]], approval: Dict[str, Any]) -> Dict[str, Any]:
    fixes = {
        "deterministic_gate_failures": [],
        "judge_required_fixes": [],
        "approval_failures": approval.get("failures", []),
    }
    if not deterministic_result.get("passed", False):
        fixes["deterministic_gate_failures"] = deterministic_result.get("gates", [])

    if judge_results:
        for judge_name, result in judge_results.items():
            if isinstance(result, dict):
                fixes["judge_required_fixes"].append({
                    "judge": judge_name,
                    "approved": result.get("approved"),
                    "required_fixes": result.get("required_fixes", []),
                    "summary": result.get("summary", ""),
                })
    return fixes


def revise_section_minimally(
    section_name: str,
    draft_markdown: str,
    claims_register: Dict[str, Any],
    deterministic_result: Dict[str, Any],
    judge_results: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
) -> Dict[str, Any]:
    context = build_writer_context(section_name)
    fix_instructions = collect_fix_instructions(deterministic_result, judge_results, approval)

    reviser_context = {
        "section_name": section_name,
        "current_draft_markdown": draft_markdown,
        "current_claims_register": claims_register,
        "fix_instructions": fix_instructions,
        "allowed_context": context,
        "hard_rules": [
            "Revise minimally.",
            "Do not add new facts or claims.",
            "Do not mention missing requirements, missing payload, unavailable data, or synthetic data in the report.",
            "Remove unsupported claims rather than inventing support.",
            "Use only evidence_items already provided.",
            "Return JSON only with keys: section_name, revised_markdown, revision_notes.",
        ],
    }

    system = "You are a minimal IFRS disclosure reviser. Return JSON only."
    user = f"""
Revise the section to fix the listed issues.

Context:
{truncate_context(reviser_context)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["minimal_reviser"],
        temperature=0.05,
        max_tokens=6000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw)
    obj.setdefault("section_name", section_name)
    obj.setdefault("revised_markdown", draft_markdown)
    return obj

## Section pipeline loop

The loop writes one section, builds its claims register, runs deterministic gates, runs LLM judges only when the deterministic gates pass, and revises up to `MAX_REVISION_LOOPS`.

In [18]:
# ============================================================
# CELL 18 — RUN ONE SECTION PIPELINE
# ============================================================


def save_section_iteration(
    section_name: str,
    iteration: int,
    draft: Dict[str, Any],
    claims: Dict[str, Any],
    deterministic: Dict[str, Any],
    judges: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
):
    slug = SECTION_SLUGS[section_name]
    prefix = f"{slug}_iter{iteration}"
    write_text(draft.get("draft_markdown", draft.get("revised_markdown", "")), DIRS["drafts"] / f"{prefix}.md")
    write_json(draft, DIRS["drafts"] / f"{prefix}.json")
    write_json(claims, DIRS["claims"] / f"claims_{prefix}.json")
    write_json(deterministic, DIRS["gates"] / f"gates_{prefix}.json")
    if judges is not None:
        write_json(judges, DIRS["judges"] / f"judges_{prefix}.json")
    write_json(approval, DIRS["audit_logs"] / f"approval_{prefix}.json")


def same_issue_signature(approval: Dict[str, Any]) -> str:
    return json.dumps(approval.get("failures", approval.get("required_fixes", [])), sort_keys=True, ensure_ascii=False)[:2000]


def run_section_pipeline(section_name: str) -> Dict[str, Any]:
    print("=" * 100)
    print("SECTION:", section_name)
    print("=" * 100)

    previous_issue_signature = None
    repeated_issue_count = 0

    draft = write_section_draft(section_name)
    draft_markdown = draft.get("draft_markdown", "")

    for iteration in range(0, MAX_REVISION_LOOPS + 1):
        print(f"Iteration {iteration} — building claims register...")
        claims = build_claims_register(section_name, draft_markdown)

        print(f"Iteration {iteration} — deterministic gates...")
        deterministic = run_deterministic_gates(section_name, draft_markdown, claims)

        judges = None
        if deterministic["passed"]:
            print(f"Iteration {iteration} — LLM judges...")
            judges = run_llm_judges(section_name, draft_markdown, claims)
        else:
            print(f"Iteration {iteration} — deterministic gates failed, skipping LLM judges.")

        approval = composite_approval_gate(section_name, deterministic, judges)
        save_section_iteration(section_name, iteration, {"section_name": section_name, "draft_markdown": draft_markdown}, claims, deterministic, judges, approval)

        print("Approval:", approval.get("approved"), approval.get("scores", approval.get("reason", "")))

        if approval.get("approved"):
            slug = SECTION_SLUGS[section_name]
            approved_md_path = DIRS["approved"] / f"approved_{slug}.md"
            approved_json_path = DIRS["approved"] / f"approved_{slug}.json"
            write_text(draft_markdown, approved_md_path)
            write_json({
                "section_name": section_name,
                "status": "approved",
                "draft_markdown": draft_markdown,
                "claims_register": claims,
                "coverage_matrix_path": str(DIRS["coverage"] / f"coverage_matrix_{slug}.json"),
                "missing_requirements_path": str(DIRS["missing_requirements"] / f"missing_requirements_{slug}.json"),
                "approval": approval,
            }, approved_json_path)
            return {
                "section_name": section_name,
                "status": "approved",
                "approved_markdown_path": str(approved_md_path),
                "approved_json_path": str(approved_json_path),
                "iterations": iteration,
                "approval": approval,
            }

        sig = same_issue_signature(approval)
        if sig == previous_issue_signature:
            repeated_issue_count += 1
        else:
            repeated_issue_count = 0
        previous_issue_signature = sig

        if repeated_issue_count >= 1:
            print("Same issue repeated. Escalating to human_review.")
            break

        if iteration >= MAX_REVISION_LOOPS:
            print("Max revision loops reached. Escalating to human_review.")
            break

        print(f"Iteration {iteration} — revising minimally...")
        revised = revise_section_minimally(section_name, draft_markdown, claims, deterministic, judges, approval)
        draft_markdown = revised.get("revised_markdown", draft_markdown)
        write_json(revised, DIRS["revisions"] / f"revision_{SECTION_SLUGS[section_name]}_iter{iteration}.json")

    slug = SECTION_SLUGS[section_name]
    review_path = DIRS["approved"] / f"human_review_{slug}.md"
    write_text(draft_markdown, review_path)
    return {
        "section_name": section_name,
        "status": "human_review",
        "markdown_path": str(review_path),
        "approval": approval,
    }

In [19]:
# ============================================================
# CELL 19 — RUN ALL SECTIONS
# ============================================================

# To test a single section, set SECTION_TO_RUN in .env, e.g. SECTION_TO_RUN=Governance
SECTION_TO_RUN = os.getenv("SECTION_TO_RUN", "").strip()
sections_to_run = [SECTION_TO_RUN] if SECTION_TO_RUN else SECTIONS

section_results = []
for section in sections_to_run:
    if section not in SECTIONS:
        raise ValueError(f"Unknown section: {section}")
    result = run_section_pipeline(section)
    section_results.append(result)

write_json(section_results, OUTPUT_DIR / "section_generation_results.json")
display(pd.DataFrame(section_results))

SECTION: General Requirements


RuntimeError: Azure strong agent HTTP 404 Resource not found.
Endpoint: https://eyq***or.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gp***/chat/completions?api-version=...

This usually means the full deployment URL is wrong, the deployment name is wrong, or the deployment is not in that Azure OpenAI resource.

Check these .env variables:
AZURE_OPENAI_GPT52_DEPLOYMENT_URL
AZURE_OPENAI_FAST_DEPLOYMENT_URL

## Whole-report connectivity judge

Run this after all sections are approved. It checks consistency across sections before PDF assembly.

In [ ]:
# ============================================================
# CELL 20 — WHOLE-REPORT CONNECTIVITY JUDGE
# ============================================================


def load_approved_sections() -> Dict[str, str]:
    approved = {}
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        path = DIRS["approved"] / f"approved_{slug}.md"
        if path.exists():
            approved[section] = read_text(path)
    return approved


def run_connectivity_judge() -> Dict[str, Any]:
    approved_sections = load_approved_sections()
    if len(approved_sections) < 2:
        result = {
            "approved": False,
            "reason": "Not enough approved sections to run connectivity judge.",
            "approved_section_count": len(approved_sections),
        }
        write_json(result, DIRS["connectivity"] / "connectivity_judge_result.json")
        return result

    context = {
        "approved_sections": approved_sections,
        "checks": [
            "Terminology consistency across sections.",
            "Time horizon consistency across Strategy and Risk Management.",
            "Targets in Strategy must not contradict Metrics and Targets.",
            "Governance oversight described in Governance must align with Strategy/Risk Management references.",
            "No duplicated or contradictory claims.",
            "No missing-payload/synthetic-data limitation wording in report prose.",
        ],
    }
    system = "You are a whole-report IFRS S1/S2 connectivity judge. Return JSON only."
    user = f"""
Review the approved sections for cross-section consistency.

Return JSON with:
- approved
- connectivity_score_0_to_10
- contradictions
- terminology_issues
- target_metric_mismatches
- required_fixes
- summary

Context:
{truncate_context(context, max_chars=90000)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["whole_report_connectivity_judge"],
        temperature=0,
        max_tokens=5000,
        response_format={"type": "json_object"},
    )
    result = parse_json_response(raw)
    write_json(result, DIRS["connectivity"] / "connectivity_judge_result.json")
    return result

connectivity_result = run_connectivity_judge()
display(pd.DataFrame([connectivity_result]))

## Final Markdown and PDF handoff package

This notebook does not use the PDF layout guide for drafting. It creates an approved Markdown report and a handoff manifest for the separate PDF assembly stage.

In [ ]:
# ============================================================
# CELL 21 — BUILD FINAL MARKDOWN + PDF HANDOFF MANIFEST
# ============================================================


def assemble_final_markdown() -> Tuple[str, Path]:
    approved_sections = load_approved_sections()
    lines = []
    lines.append("# IFRS S1/S2 Sustainability-Related Financial Disclosures")
    lines.append("")
    lines.append("<!-- Generated from evidence-supported synthetic payloads. Missing requirements are stored separately in JSON audit files and are not disclosed in report prose. -->")
    lines.append("")

    for idx, section in enumerate(SECTIONS, start=1):
        if section not in approved_sections:
            continue
        lines.append(f"# {idx}. {section}")
        lines.append("")
        lines.append(approved_sections[section].strip())
        lines.append("")

    final_md = "\n".join(lines).strip() + "\n"
    path = DIRS["handoff"] / "approved_report_markdown.md"
    write_text(final_md, path)
    return final_md, path

final_markdown, final_markdown_path = assemble_final_markdown()

handoff_manifest = {
    "pipeline_mode": PIPELINE_MODE,
    "approved_report_markdown": str(final_markdown_path),
    "approved_sections_dir": str(DIRS["approved"]),
    "coverage_dir": str(DIRS["coverage"]),
    "missing_requirements_dir": str(DIRS["missing_requirements"]),
    "claims_registers_dir": str(DIRS["claims"]),
    "connectivity_judge_result": str(DIRS["connectivity"] / "connectivity_judge_result.json"),
    "rendering_layout_guide": str(RENDERING_DIR / "layout_style_guide.json"),
    "important_rule": "The PDF assembly stage may use layout_style_guide.json. Drafting agents must not use it.",
}
write_json(handoff_manifest, DIRS["handoff"] / "pdf_handoff_manifest.json")

print("Final Markdown:", final_markdown_path)
print("PDF handoff manifest:", DIRS["handoff"] / "pdf_handoff_manifest.json")

In [ ]:
# ============================================================
# CELL 22 — AUDIT SUMMARY
# ============================================================

summary = {
    "pipeline_mode": PIPELINE_MODE,
    "forbid_invention": FORBID_INVENTION,
    "allow_partial_coverage": ALLOW_PARTIAL_COVERAGE,
    "sections": {},
    "outputs": {name: str(path) for name, path in DIRS.items()},
}

for section in SECTIONS:
    slug = SECTION_SLUGS[section]
    coverage = coverage_by_section.get(section, [])
    missing = missing_registers_by_section.get(section, {}).get("missing_requirements", [])
    approved_path = DIRS["approved"] / f"approved_{slug}.md"
    summary["sections"][section] = {
        "requirements_total": len(requirements_by_section.get(section, [])),
        "coverage_counts": dict(Counter([c["coverage_status"] for c in coverage])),
        "missing_requirements_count": len(missing),
        "approved_markdown_exists": approved_path.exists(),
        "approved_markdown_path": str(approved_path) if approved_path.exists() else None,
        "missing_requirements_path": str(DIRS["missing_requirements"] / f"missing_requirements_{slug}.json"),
    }

write_json(summary, OUTPUT_DIR / "generation_audit_summary.json")

summary_md = [
    "# Agentic IFRS Report Generation Audit Summary",
    "",
    f"- Pipeline mode: `{PIPELINE_MODE}`",
    f"- Forbid invention: `{FORBID_INVENTION}`",
    f"- Allow partial coverage: `{ALLOW_PARTIAL_COVERAGE}`",
    "",
    "## Policy",
    "",
    "The report contains only evidence-supported disclosures. Requirements not available in the synthetic payload are recorded in JSON audit files and are not mentioned in report prose.",
    "",
    "## Section summary",
    "",
]

for section, info in summary["sections"].items():
    summary_md.append(f"### {section}")
    summary_md.append(f"- Requirements total: {info['requirements_total']}")
    summary_md.append(f"- Coverage counts: `{info['coverage_counts']}`")
    summary_md.append(f"- Missing requirements count: {info['missing_requirements_count']}")
    summary_md.append(f"- Approved markdown exists: {info['approved_markdown_exists']}")
    summary_md.append("")

write_text("\n".join(summary_md), OUTPUT_DIR / "generation_audit_summary.md")
print("Saved audit summary:", OUTPUT_DIR / "generation_audit_summary.md")
display(pd.DataFrame(summary["sections"]).T)